# Customer Acquisition Response Predictor

A bank ran phone campaigns offering customers a term deposit. For each call we know who the
customer was, how and when they were contacted, and the economic conditions at the time. We also
know whether they said **yes**.

This notebook does three things:

1. **Predicts who will say yes**, using two models: a simple Logistic Regression and XGBoost.
2. **Explains what drives a yes**, using XGBoost feature importance.
3. **Groups customers into segments** with K-Means, so a marketing team can decide who to target.

In a customer-acquisition job the question is the same: *out of everyone we could contact, who is
worth the cost of contacting?*

**Dataset:** UCI Bank Marketing, `bank-additional-full.csv` (41,188 calls, May 2008 to Nov 2010).
Moro, Cortez & Rita (2014), *A Data-Driven Approach to Predict the Success of Bank Telemarketing*.

## 0. Setup

We import the libraries and fix the **random seed**. Several steps involve randomness: the
train/test split, XGBoost's tree building, and K-Means' starting points. Fixing the seed means
anyone who runs this notebook gets exactly the same numbers. That is what makes the results
reproducible.

In [ ]:
import io
import zipfile
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix, ConfusionMatrixDisplay)
from xgboost import XGBClassifier

SEED = 42
np.random.seed(SEED)

# Chart styling: a quiet grid and a fixed colour per role, so "Logistic Regression" or
# "Subscribed" is the same colour in every chart.
sns.set_theme(style="whitegrid", rc={"grid.color": "#e6e5e0", "axes.edgecolor": "#c3c2b7"})
BLUE, ORANGE, AQUA, YELLOW, MAGENTA = "#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"
pd.set_option("display.float_format", "{:.3f}".format)
def PCT():
    '''Axis formatter showing values as percentages (a fresh one per axis, as matplotlib requires).'''
    return PercentFormatter(xmax=1)

## 1. Load the data

We download the official zip from the UCI Machine Learning Repository. The zip contains
**another** zip (`bank-additional.zip`), and the CSV we want is inside that. We save the CSV to a
local `data/` folder so later runs don't download it again. (`data/` is in `.gitignore`, so the
raw data never gets committed.)

If the download fails, the cell stops with an error. It never falls back to made-up data.

Note: the file uses **semicolons** as separators, not commas, so we pass `sep=";"`.

In [ ]:
DATA_DIR = Path("data")
CSV_PATH = DATA_DIR / "bank-additional-full.csv"
UCI_URL = "https://archive.ics.uci.edu/static/public/222/bank+marketing.zip"

if not CSV_PATH.exists():
    DATA_DIR.mkdir(exist_ok=True)
    try:
        with urllib.request.urlopen(UCI_URL, timeout=60) as resp:
            outer = zipfile.ZipFile(io.BytesIO(resp.read()))
    except Exception as e:
        raise RuntimeError(f"Could not download the UCI dataset from {UCI_URL}: {e}") from e
    # The outer zip holds bank-additional.zip; the CSV lives inside that inner zip.
    inner = zipfile.ZipFile(io.BytesIO(outer.read("bank-additional.zip")))
    CSV_PATH.write_bytes(inner.read("bank-additional/bank-additional-full.csv"))

df = pd.read_csv(CSV_PATH, sep=";")
print(df.shape)
df.head()

## 2. Exploratory data analysis (EDA)

Before modelling, we look at the data: what columns there are, whether anything is missing, and
how the target is distributed.

The columns fall into four groups:

- **Customer:** `age`, `job`, `marital`, `education`, `default`, `housing` (has a housing loan),
  `loan` (has a personal loan)
- **This campaign's contact:** `contact` (cellular/telephone), `month`, `day_of_week`,
  `duration` (call length in seconds), `campaign` (number of calls to this customer in this campaign)
- **Previous campaigns:** `pdays` (days since last contacted; **999 means never**), `previous`
  (number of earlier contacts), `poutcome` (result of the previous campaign)
- **Economy at the time:** `emp.var.rate`, `cons.price.idx`, `cons.conf.idx`, `euribor3m`,
  `nr.employed`

The target `y` is `yes` or `no`.

In [ ]:
df.info()

The dataset has no `NaN` values. Some categorical columns do use the word `"unknown"` for missing
answers, though. We keep `"unknown"` as its own category, because *not knowing* someone's
education can itself carry information.

In [ ]:
# Count how often "unknown" appears in each text column
cat_cols_all = df.select_dtypes(exclude="number").columns
(df[cat_cols_all] == "unknown").sum().loc[lambda s: s > 0]

### 2.1 Class balance: the target is imbalanced

This is the most important fact about the dataset. Most customers said **no**, so a "model" that
always predicts "no" would already score high **accuracy**. For that reason we won't judge models
on accuracy alone. We'll also look at **precision, recall, F1 and ROC-AUC** (defined in section 4).

In [ ]:
counts = df["y"].value_counts()
rate = df["y"].eq("yes").mean()
print(counts.to_string())
print(f"\nSubscription rate: {rate:.1%}")

fig, ax = plt.subplots(figsize=(5, 3.2))
bars = ax.bar(counts.index, counts.values, color=[BLUE, ORANGE], width=0.55)
ax.bar_label(bars, labels=[f"{v:,} ({v/len(df):.1%})" for v in counts.values], padding=3)
ax.set_title("Target class balance (y)")
ax.set_ylabel("Customers")
ax.set_ylim(0, counts.max() * 1.15)
sns.despine()
plt.tight_layout()
plt.show()

### 2.2 A trap: the `duration` column (data leakage)

`duration` is the length of the call. It is extremely predictive: people who say yes stay on the
phone longer. **But we only know the duration after the call has ended**, and by then we also know
the answer. A marketer deciding *whom to call* can't use it.

Using a feature that won't be available at prediction time is called **data leakage**. It inflates
results on paper and fails in real use. The dataset's own documentation says to drop `duration`
for any realistic predictive model, so we do. The chart below shows why it's tempting.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
sns.boxplot(data=df, x="duration", y="y", hue="y", palette={"no": BLUE, "yes": ORANGE},
            showfliers=False, legend=False, ax=ax)  # hide outliers so the boxes are readable
ax.set_title("Call duration by outcome: strongly separated, but only known after the call")
ax.set_xlabel("Call duration (seconds)")
ax.set_ylabel("Subscribed")
sns.despine()
plt.tight_layout()
plt.show()

### 2.3 Key feature distributions

Three quick looks at features we *can* know before calling:

1. **Age:** do younger or older people respond differently?
2. **Previous campaign outcome:** did people who said yes before say yes again?
3. **Month of contact:** does timing matter?

For the two categorical charts we plot the **subscription rate** per group (the share who said
yes), not raw counts. A group can be small but respond well. Group sizes appear in the labels.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.4))
sns.histplot(data=df, x="age", hue="y", bins=40, stat="density", common_norm=False,
             element="step", fill=False, linewidth=2,
             palette={"no": BLUE, "yes": ORANGE}, ax=ax)  # common_norm=False: each class sums to 1
ax.set_title("Age distribution by outcome (each class normalised)")
ax.set_xlabel("Age")
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
def rate_by(col, order=None):
    '''Subscription rate and group size for each value of a column.'''
    g = df.groupby(col)["y"].agg(rate=lambda s: s.eq("yes").mean(), n="size")
    return g.reindex(order) if order is not None else g.sort_values("rate")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6), gridspec_kw={"width_ratios": [1, 2]})

pout = rate_by("poutcome")
bars = axes[0].barh(pout.index, pout["rate"], color=BLUE, height=0.55)
axes[0].bar_label(bars, labels=[f"{r:.0%}  (n={n:,})" for r, n in zip(pout["rate"], pout["n"])], padding=3)
axes[0].set_title("Rate by previous-campaign outcome")
axes[0].set_xlim(0, pout["rate"].max() * 1.6)
axes[0].xaxis.set_major_formatter(PCT())

months = ["mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"]
mon = rate_by("month", order=months)
bars = axes[1].bar(mon.index, mon["rate"], color=BLUE, width=0.6)
axes[1].bar_label(bars, labels=[f"{r:.0%}" for r in mon["rate"]], padding=3)
axes[1].axhline(rate, color="#52514e", linestyle="--", linewidth=1)
axes[1].text(len(months) - 0.5, rate, f" overall {rate:.1%}", va="bottom", ha="right", color="#52514e")
axes[1].set_title("Rate by contact month (no calls in Jan/Feb)")
axes[1].yaxis.set_major_formatter(PCT())
axes[1].set_ylim(0, mon["rate"].max() * 1.15)
sns.despine()
plt.tight_layout()
plt.show()

print("Calls per month:\n", mon["n"].to_string())

**How to read these charts:** look for groups whose subscription rate sits well above the overall
rate *and* whose group is not tiny. A 50% rate from 50 people is much less reliable than a 20% rate
from 5,000.

## 3. Preprocessing

Models only understand numbers, so we need to:

1. **Drop `duration`** (leakage; see 2.2).
2. **Fix `pdays`.** The value 999 means "never contacted before". Left alone, a model would read
   999 as "a very long time ago", which is wrong. We add a yes/no column `previously_contacted`
   and set `pdays` to 0 for people who were never contacted, so the number only means something
   when the flag is 1.
3. **One-hot encode categoricals.** A column like `job` becomes one 0/1 column per job
   (`job_admin.`, `job_student`, ...). We can't just number the jobs 1, 2, 3, because the model
   would then think "student" is somehow three times "admin".
4. **Scale numerics** for Logistic Regression. It compares features through their coefficients,
   so `age` (tens) and `euribor3m` (single digits) should be on the same scale. We use
   `StandardScaler`: subtract the mean, divide by the standard deviation. XGBoost is tree-based
   and doesn't need scaling, but it doesn't hurt either, so both models share one preprocessing
   step.
5. **Split into train and test (80/20), stratified.** The model learns only from the training
   set, and we grade it on the test set, which it has never seen. `stratify=y` keeps the ~11% yes
   rate the same in both halves.

**Why a `Pipeline`?** It learns the scaling and encoding **from the training data only** and then
applies them to the test data. If we scaled using the whole dataset, facts about the test set
would leak into training.

In [ ]:
data = df.drop(columns=["duration"]).copy()

# pdays = 999 is a code for "never contacted", not a real number of days
data["previously_contacted"] = (data["pdays"] != 999).astype(int)
data.loc[data["pdays"] == 999, "pdays"] = 0

X = data.drop(columns=["y"])
y = (data["y"] == "yes").astype(int)  # 1 = subscribed, 0 = did not

cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_cols = X.select_dtypes("number").columns.tolist()
print("Categorical:", cat_cols)
print("Numeric:    ", num_cols)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)
print(f"\nTrain: {X_train.shape}, yes-rate {y_train.mean():.3f}")
print(f"Test:  {X_test.shape}, yes-rate {y_test.mean():.3f}")

preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    # handle_unknown="ignore": if the test set has a category never seen in training, output all zeros instead of crashing
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
])

## 4. Model 1: Logistic Regression (the interpretable baseline)

Logistic Regression gives each feature a weight, adds them up, and squashes the total into a
probability between 0 and 1. It's simple and fast, and you can explain every prediction, which
makes it the standard **baseline**: any fancier model has to beat it to earn its complexity.

**Handling imbalance:** `class_weight="balanced"` tells the model that a mistake on a rare "yes"
costs about 8x more than a mistake on a common "no". Without it, the model would lean towards
predicting "no" for nearly everyone. We give XGBoost the same treatment, so the comparison is fair.

**The metrics** (positive class = "yes, subscribed"):

| Metric | Plain-English meaning |
|---|---|
| Accuracy | Share of all predictions that were right. Misleading when classes are imbalanced. |
| Precision | Of the people we'd call, how many actually subscribe? (Wasted calls are the cost.) |
| Recall | Of all the people who would subscribe, how many did we find? (Missed customers are the cost.) |
| F1 | A single score balancing precision and recall (their harmonic mean). |
| ROC-AUC | Probability that a random subscriber gets a higher score than a random non-subscriber. 0.5 is a coin flip, 1.0 is perfect. It doesn't depend on any threshold. |

A **confusion matrix** counts the four outcomes: true negatives, false positives, false negatives,
true positives.

In [ ]:
log_reg = Pipeline([
    ("prep", preprocess),
    ("model", LogisticRegression(class_weight="balanced", max_iter=2000, random_state=SEED)),
])
log_reg.fit(X_train, y_train)

def evaluate(name, model):
    '''Score a fitted model on the test set; return a metrics dict and the predicted probabilities.'''
    proba = model.predict_proba(X_test)[:, 1]  # column 1 = probability of "yes"
    pred = (proba >= 0.5).astype(int)          # default 0.5 cut-off
    metrics = {
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),  # uses probabilities, not the 0/1 predictions
    }
    return metrics, proba, pred

def plot_confusion(y_true, pred, title):
    fig, ax = plt.subplots(figsize=(4, 3.6))
    ConfusionMatrixDisplay(confusion_matrix(y_true, pred), display_labels=["no", "yes"]).plot(
        ax=ax, cmap="Blues", colorbar=False, values_format=",")
    ax.set_title(title)
    ax.grid(False)
    plt.tight_layout()
    plt.show()

lr_metrics, lr_proba, lr_pred = evaluate("Logistic Regression", log_reg)
print(pd.Series(lr_metrics).to_string())
plot_confusion(y_test, lr_pred, "Logistic Regression: confusion matrix")

## 5. Model 2: XGBoost

XGBoost builds hundreds of small decision trees **one after another**, and each new tree tries to
correct the mistakes of the trees before it (this is called *gradient boosting*). Trees pick up
things Logistic Regression can't, such as "young **and** contacted by cellular **and** in a bad
economy", without us having to hand-craft those combinations.

**Handling imbalance:** `scale_pos_weight` = (number of no's) / (number of yes's) in the training
set. It does the same job as `class_weight="balanced"`: each "yes" counts roughly 8x in training.

**Settings, briefly:**
- `n_estimators=400`, `learning_rate=0.05`: many small correction steps (a low learning rate is
  slower but steadier).
- `max_depth=4`: each tree asks at most 4 questions, which keeps trees simple and limits
  overfitting.
- `subsample` / `colsample_bytree=0.8`: each tree sees a random 80% of rows and columns, another
  guard against memorising the training data.

These are reasonable defaults. We didn't tune them, so a tuned model could do somewhat better.

In [ ]:
spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {spw:.2f}")

xgb = Pipeline([
    ("prep", preprocess),
    ("model", XGBClassifier(
        n_estimators=400, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=spw, eval_metric="auc",
        random_state=SEED, n_jobs=-1)),
])
xgb.fit(X_train, y_train)

xgb_metrics, xgb_proba, xgb_pred = evaluate("XGBoost", xgb)
print(pd.Series(xgb_metrics).to_string())
plot_confusion(y_test, xgb_pred, "XGBoost: confusion matrix")

## 6. Model comparison

Here are both models side by side, on the same test set and with the same 0.5 cut-off.

The **ROC curve** shows every possible cut-off at once. Moving the cut-off down catches more
subscribers (the curve moves up) but also flags more non-subscribers (it moves right). A curve
that hugs the top-left corner is better. The area under it is the ROC-AUC. The dashed diagonal
is random guessing.

In [ ]:
comparison = pd.DataFrame([lr_metrics, xgb_metrics]).set_index("Model")
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 5))
for name, proba, color in [("Logistic Regression", lr_proba, BLUE), ("XGBoost", xgb_proba, ORANGE)]:
    fpr, tpr, _ = roc_curve(y_test, proba)
    ax.plot(fpr, tpr, color=color, linewidth=2,
            label=f"{name} (AUC = {roc_auc_score(y_test, proba):.3f})")
ax.plot([0, 1], [0, 1], linestyle="--", color="#8a8984", linewidth=1, label="Random guess (AUC = 0.5)")
ax.set_xlabel("False positive rate (non-subscribers flagged)")
ax.set_ylabel("True positive rate (subscribers found)")
ax.set_title("ROC curves on the test set")
ax.legend(loc="lower right", frameon=False)
ax.set_aspect("equal")
sns.despine()
plt.tight_layout()
plt.show()

### 6.1 What does this mean for a campaign? (Lift in the top 10%)

Marketers rarely think in AUC. The practical question is: *"If we can only call 10% of the list,
how many more subscribers do we reach by calling the people the model ranks highest, instead of
calling at random?"* That ratio is called **lift**.

In [ ]:
def top_decile_lift(proba):
    cutoff = np.quantile(proba, 0.9)          # score above which the top 10% sit
    top = y_test[proba >= cutoff]
    return top.mean(), top.mean() / y_test.mean()

base = y_test.mean()
print(f"Random calling: {base:.1%} of calls convert\n")
for name, proba in [("Logistic Regression", lr_proba), ("XGBoost", xgb_proba)]:
    hit, lift = top_decile_lift(proba)
    print(f"{name:20s} top-10% conversion {hit:.1%}  ->  lift {lift:.2f}x")

## 7. What drives a response? (XGBoost feature importance)

XGBoost can report how much each feature helped its trees make better splits. We use
**gain**-based importance: the average improvement in prediction quality each time a feature is
used.

**Caveats you should be able to say in an interview:**
- Importance says a feature is **useful for prediction**, not that it **causes** subscriptions.
- Highly correlated features (the economic indicators move together) share or swap importance
  between them, so read them as one group.
- Because of one-hot encoding, each category (for example `poutcome_success`) gets its own bar.

In [ ]:
feature_names = xgb.named_steps["prep"].get_feature_names_out()
# get_feature_names_out adds prefixes like "num__age" / "cat__job_student"; strip them for readability
feature_names = [f.split("__", 1)[1] for f in feature_names]

booster = xgb.named_steps["model"].get_booster()
gain = booster.get_score(importance_type="gain")  # keys are "f0", "f1", ... in column order
importance = (pd.Series({feature_names[int(k[1:])]: v for k, v in gain.items()})
              .sort_values(ascending=False))
importance = importance / importance.sum()  # express as share of total gain

top = importance.head(15).iloc[::-1]  # reverse so the biggest bar is on top
fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.barh(top.index, top.values, color=BLUE, height=0.6)
ax.bar_label(bars, labels=[f"{v:.1%}" for v in top.values], padding=3)
ax.set_title("XGBoost: top 15 features by share of total gain")
ax.set_xlabel("Share of total gain")
ax.xaxis.set_major_formatter(PCT())
ax.set_xlim(0, top.max() * 1.2)
sns.despine()
plt.tight_layout()
plt.show()

importance.head(15)

Importance only tells us *how much* a feature matters, not *in which direction*. To check the
direction, we compare the actual subscription rate across values of the top numeric drivers
(split into quartiles) and the top categorical drivers.

In [ ]:
check = df.drop(columns=["duration"]).copy()
check["subscribed"] = check["y"].eq("yes")

for col in ["nr.employed", "euribor3m", "emp.var.rate", "age", "campaign"]:
    # qcut splits into equal-sized groups; duplicates="drop" handles columns with many identical values
    bins = pd.qcut(check[col], q=4, duplicates="drop")
    print(f"--- {col}: subscription rate by quartile ---")
    print(check.groupby(bins, observed=True)["subscribed"].agg(rate="mean", n="size").to_string(), "\n")

for col in ["poutcome", "contact"]:
    print(f"--- {col} ---")
    print(check.groupby(col)["subscribed"].agg(rate="mean", n="size").to_string(), "\n")

**Plain-English reading of the top drivers.** Read this alongside the chart and the tables above
from your own run. The directions below come from the known structure of this dataset.

- **The economy (`nr.employed`, `euribor3m`, `emp.var.rate`).** These say *when* the call
  happened more than *who* was called. Rates were far higher when employment and interest rates
  were low (the post-2008 downturn). In practice this means campaign timing matters at least as
  much as customer choice. It also means a model trained in one economic climate may not carry
  over to another.
- **Previous success (`poutcome_success`, `previously_contacted`, `pdays`).** Customers who said
  yes to a previous campaign are by far the most likely to say yes again. They are the warmest
  leads.
- **Contact channel and month (`contact_telephone`, `month_*`).** Landline contacts convert worse
  than mobile ones. Some months (March, September, October, December) have very high rates but
  few calls. May has the most calls and one of the lowest rates.
- **Number of calls this campaign (`campaign`).** Calling the same person more and more often
  shows diminishing returns.

## 8. Customer segmentation with K-Means

Prediction tells us *who* is likely to respond. Segmentation groups customers who **look alike**,
so marketing can plan a different message or channel for each group.

**K-Means** places *k* centre points, assigns every customer to the nearest centre, moves each
centre to the average of its group, and repeats until nothing changes.

**Which features?** We segment on things that describe the **customer and their history with the
bank**, the kind of information a marketer would have before a campaign:
`age`, `previous` (number of past contacts), `previously_contacted`, `prev_success` (said yes
before), `housing` loan, personal `loan`, and `campaign` (calls so far).

We deliberately leave out **`y`**. Segmentation is *unsupervised*: we group first and only then
check how each group responded. We also leave out the economic indicators, which describe the
calendar rather than the person.

K-Means measures distance, so we **scale** features first. Otherwise `age` (range ~ 60) would
drown out a 0/1 flag. We also take `log(1 + campaign)` because a handful of people were called
40+ times, and those outliers would otherwise pull a centre towards themselves.

In [ ]:
seg = pd.DataFrame({
    "age": df["age"],
    "log_campaign": np.log1p(df["campaign"]),        # log1p = log(1 + x), tames the long tail
    "previous": df["previous"],
    "previously_contacted": (df["pdays"] != 999).astype(int),
    "prev_success": (df["poutcome"] == "success").astype(int),
    "housing_loan": (df["housing"] == "yes").astype(int),
    "personal_loan": (df["loan"] == "yes").astype(int),
})
seg_scaled = StandardScaler().fit_transform(seg)
seg.describe().T[["mean", "std", "min", "max"]]

### 8.1 Choosing *k* with the elbow method

We run K-Means for k = 1 to 10 and record the **inertia**: the total squared distance from each
customer to their centre (lower means tighter groups). Inertia always falls as k grows, since
with 41,000 clusters it would be zero. We look for the **elbow**, the point after which extra
clusters buy much less improvement.

Picking the elbow by eye is subjective. To keep it reproducible, we also compute it: draw a
straight line from the first point to the last, and pick the k whose point is **furthest below
that line**.

In [ ]:
ks = range(1, 11)
inertias = [KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(seg_scaled).inertia_ for k in ks]

# Knee detection: normalise both axes to 0-1, then find the point furthest from the first-to-last line
x = np.array(ks, dtype=float); y_in = np.array(inertias)
xn = (x - x.min()) / (x.max() - x.min())
yn = (y_in - y_in.min()) / (y_in.max() - y_in.min())
distance = (1 - xn) - yn   # the line runs from (0,1) to (1,0), i.e. y = 1 - x
best_k = int(x[np.argmax(distance)])
print(f"Elbow at k = {best_k}")

fig, ax = plt.subplots(figsize=(6, 3.4))
ax.plot(list(ks), inertias, marker="o", color=BLUE, linewidth=2, markersize=7)
ax.plot(best_k, inertias[best_k - 1], marker="o", markersize=12, markerfacecolor="none",
        markeredgecolor=ORANGE, markeredgewidth=2)
ax.annotate(f"elbow: k = {best_k}", (best_k, inertias[best_k - 1]),
            textcoords="offset points", xytext=(12, 10), color="#0b0b0b")
ax.set_xticks(list(ks))
ax.set_xlabel("Number of clusters (k)")
ax.set_ylabel("Inertia")
ax.set_title("Elbow method")
sns.despine()
plt.tight_layout()
plt.show()

### 8.2 Fit the final model and profile each segment

For each segment we report its size, the average of each feature, and the **actual subscription
rate** (which K-Means never saw). The one-line descriptions are generated from each segment's
profile, comparing it with the overall average.

In [ ]:
kmeans = KMeans(n_clusters=best_k, n_init=10, random_state=SEED).fit(seg_scaled)
df["segment"] = kmeans.labels_

profile = (seg.assign(segment=kmeans.labels_, subscribed=df["y"].eq("yes"),
                      campaign=df["campaign"])
              .groupby("segment")
              .agg(customers=("age", "size"), age=("age", "mean"), calls=("campaign", "mean"),
                   previous=("previous", "mean"), prev_contacted=("previously_contacted", "mean"),
                   prev_success=("prev_success", "mean"), housing_loan=("housing_loan", "mean"),
                   personal_loan=("personal_loan", "mean"), subscription_rate=("subscribed", "mean")))
profile["share"] = profile["customers"] / profile["customers"].sum()
profile = profile.sort_values("subscription_rate", ascending=False)
profile

In [ ]:
overall = {c: profile[c].mul(profile["customers"]).sum() / profile["customers"].sum()
           for c in ["age", "calls", "prev_contacted", "prev_success", "housing_loan",
                     "personal_loan", "subscription_rate"]}

def describe(row):
    '''One-line, rule-based description of a segment relative to the overall average.'''
    parts = []
    if row.prev_success > 0.5:
        parts.append("past campaign successes")
    elif row.prev_contacted > 0.5:
        parts.append("previously contacted, mostly without success")
    else:
        parts.append("never contacted before")
    if row.age >= overall["age"] + 8:
        parts.append(f"older (avg age {row.age:.0f})")
    elif row.age <= overall["age"] - 6:
        parts.append(f"younger (avg age {row.age:.0f})")
    else:
        parts.append(f"mid-age (avg {row.age:.0f})")
    if row.personal_loan > 0.5:
        parts.append("holds a personal loan")
    parts.append("has a housing loan" if row.housing_loan > 0.5 else "no housing loan")
    if row.calls >= overall["calls"] * 1.5:
        parts.append(f"called often this campaign ({row.calls:.1f} times)")
    lift = row.subscription_rate / overall["subscription_rate"]
    return (f"{', '.join(parts)}. {row.customers:,.0f} customers ({row.share:.0%}), "
            f"subscription rate {row.subscription_rate:.1%} ({lift:.1f}x average)")

for seg_id, row in profile.iterrows():
    print(f"Segment {seg_id}: {describe(row)}")

### 8.3 Segment chart

The data has 7 features, which we can't plot directly. **PCA** (Principal Component Analysis)
squeezes them into 2 new axes that keep as much of the spread between customers as possible.
We plot a random sample of 4,000 customers to keep the chart readable. Each segment gets its own
colour **and** marker shape, so segments can be told apart without relying on colour alone.

Because several features are 0/1 flags, the points form bands and stripes. That is expected.

In [ ]:
pca = PCA(n_components=2, random_state=SEED)
coords = pca.fit_transform(seg_scaled)
print(f"The 2 PCA axes keep {pca.explained_variance_ratio_.sum():.0%} of the variation")

rng = np.random.default_rng(SEED)
idx = rng.choice(len(coords), size=4000, replace=False)

palette = [BLUE, ORANGE, AQUA, YELLOW, MAGENTA, "#008300", "#4a3aa7", "#e34948", "#52514e", "#8a8984"]
markers = ["o", "s", "^", "D", "v", "P", "X", "*", "<", ">"]
fig, ax = plt.subplots(figsize=(7.5, 5.5))
for s in sorted(df["segment"].unique()):
    m = idx[df["segment"].values[idx] == s]
    ax.scatter(coords[m, 0], coords[m, 1], s=18, alpha=0.55, color=palette[s], marker=markers[s],
               edgecolor="white", linewidth=0.4,
               label=f"Segment {s} ({profile.loc[s, 'subscription_rate']:.0%} subscribe)")
ax.set_xlabel(f"PCA 1 ({pca.explained_variance_ratio_[0]:.0%} of variation)")
ax.set_ylabel(f"PCA 2 ({pca.explained_variance_ratio_[1]:.0%} of variation)")
ax.set_title(f"Customer segments (K-Means, k = {best_k}), 4,000-customer sample")
ax.legend(frameon=False, loc="best", markerscale=1.6)
sns.despine()
plt.tight_layout()
plt.show()

In [ ]:
# Subscription rate by segment: the "so what" of segmentation
order = profile.index
fig, ax = plt.subplots(figsize=(6.5, 0.6 + 0.55 * len(order)))
bars = ax.barh([f"Segment {s}" for s in order][::-1], profile["subscription_rate"].values[::-1],
               color=BLUE, height=0.55)
ax.bar_label(bars, labels=[f"{r:.1%}  (n={n:,})" for r, n in
             zip(profile["subscription_rate"].values[::-1], profile["customers"].values[::-1])], padding=3)
ax.axvline(overall["subscription_rate"], color="#52514e", linestyle="--", linewidth=1)
ax.set_title("Subscription rate by segment (dashed = overall)")
ax.xaxis.set_major_formatter(PCT())
ax.set_xlim(0, profile["subscription_rate"].max() * 1.45)
sns.despine()
plt.tight_layout()
plt.show()

## 9. Results summary

This cell prints the key numbers from this run in one place (these are the numbers used in the
README).

In [ ]:
print("=== Model comparison (test set, n = {:,}) ===".format(len(y_test)))
print(comparison.round(3).to_string())
print("\n=== Top-10% lift ===")
for name, proba in [("Logistic Regression", lr_proba), ("XGBoost", xgb_proba)]:
    hit, lift = top_decile_lift(proba)
    print(f"{name:20s} {hit:.1%} conversion, {lift:.2f}x lift")
print("\n=== Top 5 XGBoost features (share of gain) ===")
print(importance.head(5).round(3).to_string())
print(f"\n=== Segments (k = {best_k}) ===")
for seg_id, row in profile.iterrows():
    print(f"Segment {seg_id}: {describe(row)}")

## 10. So what?

**Which model to use.** Pick whichever has the higher ROC-AUC and top-10% lift in the summary
above, and only if the gap is large enough to matter. On this dataset, with `duration` removed,
the two models usually land close together. When the gap is small, Logistic Regression is the
better business choice: it's cheaper to run and easier to explain to compliance and marketing,
and its coefficients can be read directly. XGBoost earns its place only if its extra lift pays
for that. Either way, **don't use the default 0.5 cut-off blindly.** Set it by budget, for
example "call the top 10% of scores", and use the lift figure to estimate the return.

**Which segments to target.** Start with the segment(s) whose subscription rate is well above
the average line in the chart above, typically customers who **said yes to a previous
campaign**. Then, within the large, low-rate segments, use the model's score to pick the best
individuals instead of calling the whole group. Stop calling people who have already been called
many times this campaign.

**Honest limitations**
- **`duration` was removed** because it's only known after the call. Published results that
  include it (AUC ≈ 0.93+) are not realistic for deciding whom to call. Expect lower but honest
  numbers here.
- **Economic features dominate.** The data covers 2008 to 2010, a single unusual economic period.
  The model partly learns *when* the calls happened, which won't transfer cleanly to a different
  economy. A time-based split (train on early months, test on later ones) would be a stricter test
  than our random split.
- **No hyperparameter tuning or cross-validation.** The numbers come from a single 80/20 split.
  They are honest but carry some sampling noise.
- **One bank, one product, one country (Portugal).** The results say nothing directly about
  credit-card acquisition at another company. The *workflow* transfers; the coefficients don't.
- **K-Means assumes round, similar-sized clusters** and treats 0/1 flags as distances. The segments
  are a useful summary, not ground truth. Different feature choices would give different segments.
- **Feature importance is predictive, not causal.** "Previous success" predicts a yes. It doesn't
  mean contacting someone *makes* them say yes.